SHAP

In [5]:
import pandas as pd
import lightgbm as lgb
import numpy as np

file_path = "../data/processed/market_data_clean_v2.parquet"
d = pd.read_parquet(file_path)
d = d.drop(columns=['naew'])
d.columns = [c.lower() for c in d.columns]
d = d.sort_values(['product_name', 'month_idx']).reset_index(drop=True)   # moved up, before split
# print("File read, Total Rows: ", len(d))

train = d[d['month_idx'] <= 8].copy()
valid = d[d['month_idx'] == 9].copy()
test  = d[d['month_idx'] == 10].copy()
# print("\n Train Val Test Split: ", len(train), len(valid), len(test))

exclude = ['product_name','category','bs_year','bs_month','month_idx','month_name',
           'volume','min_price','max_price','avg_price','unit','unit_canonical',
           'unit_changed','total_amount','volume_equals','total_sources',
           'reconciliation_gap','import_share','domestic_share','n_months_present','is_balanced',
           'india_share','china_share','bhutan_share','n_sources','herfindahl',
           'm_sin','m_cos','avg_price_lag1','volume_lag1']

src_cols = [c for c in d.columns if c not in exclude]
imp_cols = [c for c in ['india', 'china', 'bhutan'] if c in src_cols]

feature_cols = (
    ['product_name', 'category', 'unit', 'm_sin', 'm_cos', 'n_sources',
     'herfindahl', 'import_share', 'domestic_share', 'n_months_present',
     'india_share', 'china_share', 'bhutan_share', 'avg_price_lag1']
    + src_cols
)
cat_cols = ['product_name', 'category', 'unit']
target_col = 'avg_price'

for df_ in [train, valid, test]:
    for c in cat_cols:
        df_[c] = df_[c].astype('category')


import shap

# ---- explain on the TEST month only, as required ----
booster_for_shap = lgb.Booster(model_file='../models/price_surrogate_v1.txt')  # train-only model, matches your honest test evaluation

X_test_shap = test[feature_cols].copy()
for c in cat_cols:
    X_test_shap[c] = X_test_shap[c].astype('category')

explainer = shap.TreeExplainer(booster_for_shap)
sv_test = explainer(X_test_shap)

In [ ]:
import matplotlib.pyplot as plt

# ---- 1. global bar ----
shap.plots.bar(sv_test, show=False)
plt.tight_layout()
plt.savefig('../outputs/images/shap_bar_test_price_train.png', dpi=150)
plt.show()

# ---- 2. beeswarm ----
shap.plots.beeswarm(sv_test, show=False)
plt.tight_layout()
plt.savefig('../outputs/images/shap_beeswarm_test_price_train.png', dpi=150)
plt.show()

# ---- 3. dependence plot on import_share (your key story feature) ----
shap.plots.scatter(sv_test[:, 'import_share'], show=False)
plt.tight_layout()
plt.savefig('../outputs/images/shap_dependence_import_share_price_train.png', dpi=150)
plt.show()

# ---- 4. local waterfall, one example row ----
shap.plots.waterfall(sv_test[0], show=False)
plt.tight_layout()
plt.savefig('../outputs/images/shap_waterfall_test_example_price_train.png', dpi=150)
plt.show()

Now the fifth, role-specific one - before/after a shock, using price_surrogate_final

In [ ]:
from features import add_derived_features
from scenario_engine004 import build_scenario_baseline, get_product_baseline


booster_prod = lgb.Booster(model_file='../models/price_surrogate_final.txt')
explainer_prod = shap.TreeExplainer(booster_prod)

# pick a product with a real, visible shock effect — Jack_Fruit or Cabbage from your ranking
product = 'Jack_Fruit'
baseline_row, _ = build_scenario_baseline(d, product)
X_base = baseline_row[feature_cols].copy()
for c in cat_cols:
    X_base[c] = X_base[c].astype('category')

base_val, _, _ = get_product_baseline(d, product, 'india')
shocked_row = baseline_row.copy()
shocked_row['india'] = base_val * 0.7
shocked_row = add_derived_features(shocked_row, src_cols, imp_cols)
X_shock = shocked_row[feature_cols].copy()
for c in cat_cols:
    X_shock[c] = X_shock[c].astype('category')

sv_base = explainer_prod(X_base)
sv_shock = explainer_prod(X_shock)

fig, axes = plt.subplots(1, 2, figsize=(16,6))
plt.sca(axes[0]); shap.plots.waterfall(sv_base[0], show=False); axes[0].set_title(f'{product} — baseline')
plt.sca(axes[1]); shap.plots.waterfall(sv_shock[0], show=False); axes[1].set_title(f'{product} — after India −30% shock')
plt.tight_layout()
plt.savefig('../outputs/images/shap_waterfall_baseline_vs_shocked_price_final.png', dpi=150)
plt.show()

model train vs test

In [ ]:
pred_test_price = np.expm1(booster_eval.predict(X_test_eval))
actual_test_price = test[target_col].values
fig, ax = plt.subplots(figsize=(7,7))
ax.scatter(actual_test_price, pred_test_price, alpha=0.6, edgecolor='k', linewidth=0.3)
lims = [0, max(actual_test_price.max(), pred_test_price.max())*1.05]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual price'); ax.set_ylabel('Predicted price')
ax.set_title(f'Test set: Actual vs Predicted (R²={r2_test:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/images/test_actual_vs_predicted.png', dpi=150)
plt.show()

Risk Signal Distribution

In [ ]:
signal_cols = ['volatility', 'import_share', 'concentration', 'volume_cv']
v3 = pd.read_parquet('../data/processed/market_data_clean_v3.parquet')

fig, axes = plt.subplots(2, 2, figsize=(11,8))
for ax, col in zip(axes.flat, signal_cols):
    ax.hist(v3[col].dropna(), bins=30, color='#2c7fb8', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
plt.suptitle('Raw risk signal distributions (before min-max scaling)')
plt.tight_layout()
plt.savefig('../outputs/images/risk_signal_distributions.png', dpi=150)
plt.show()

risk_category_by_product

In [ ]:
risk_counts = v3.groupby(['category', 'risk'], observed=True).size().unstack(fill_value=0)
risk_counts = risk_counts[['Low','Medium','High']]  # consistent order

fig, ax = plt.subplots(figsize=(7,5))
risk_counts.plot(kind='bar', stacked=True, ax=ax, color=['#2c7fb8','#fdae61','#d7191c'])
ax.set_ylabel('Number of product-months')
ax.set_title('Risk category distribution by product category')
plt.tight_layout()
plt.savefig('../outputs/images/risk_category_by_product.png', dpi=150)
plt.show()

scenario shock

In [ ]:
 # move to 004 scenario test 